# 02 · Entrenamiento — fine-tuning de FractalMAR-Base (256 px) en 15 GB

**Idea central:** en vez de entrenar desde cero (lo que daba pérdida alta), se parte de **FractalMAR-Base preentrenado en ImageNet-256 (186 M de parámetros)** del repo oficial y se **ajusta (fine-tuning)** a tus 3 órganos.

**Qué se usa y por qué**
| Elemento | Qué es | Por qué |
|---|---|---|
| `fractalmar_base_in256` (repo oficial) | 4 niveles fractales 256→16→4→1; MAR en los 3 primeros niveles y AR por canal RGB en el último | modelo consolidado; no reimplemento nada |
| Preentrenado ImageNet-256 | pesos de partida (712 MB, se descarga una vez y se guarda en Drive) | con ~4 mil imágenes, empezar desde cero converge mal; aquí solo se adapta |
| `class_emb` nuevo (3 clases) | única capa que cambia de forma (1000→3) | se inicia con la media de ImageNet + ruido pequeño |
| `guiding_pixel=True`, `num_conds=5`, `r_weight=5` | configuración que el repo usa a 256 px | el notebook de 64 px usaba `r_weight=1` y `guiding_pixel=False` (válido a 64, no a 256) |
| fp16 + `GradScaler` | precisión mixta | la T4 no soporta bf16; es lo que usa el repo |
| `grad_checkpointing=True` | recalcula activaciones en el backward | la memoria de activaciones es lo que no cabe en 15 GB |
| micro-batch automático + acumulación | batch lógico 32 = micro × acumulación | el notebook prueba cuál micro-batch cabe y lo usa |
| LR 2e-5, warmup 100, coseno | fine-tuning conservador | LR alto destruye lo preentrenado |
| Aumentos D4 (flips + rot. 90°) por imagen | el tejido no tiene orientación | multiplica ×8 los datos efectivos sin cambiar la estadística |
| Muestreo balanceado por órgano | cada órgano aparece igual de seguido | Lung tiene menos imágenes |
| Checkpoints en Drive + reanudable | `last.pth` (completo) y `best_fp16.pth` (por val loss) | Colab se desconecta: **volver a ejecutar continúa donde quedó** |

**Memoria (aprox.):** pesos 0.75 GB + gradientes 0.75 GB + AdamW 1.5 GB ≈ 3 GB fijos; el resto son activaciones, que se controlan con el micro-batch.

**Cómo leer las pérdidas:** la pérdida es entropía cruzada sobre 256 niveles de intensidad por sub-píxel (adivinar al azar = ln 256 ≈ 5.55). **No** va a ir a 0; lo que importa es que la **val loss baje** y que no se separe mucho de la de entrenamiento (sobreajuste).

In [ ]:
import os, sys
from google.colab import drive
drive.mount("/content/drive")
sys.path.insert(0, "/content/drive/MyDrive/fractal_histopatologia_256/code")   # carpeta con fg_*.py

!pip -q install "timm==0.9.12"
import numpy as np, torch
import fg_config as cfg, fg_common as fg
fg.mount_drive(); fg.setup_repo(); fg.seed_everything()
total_gb = fg.gpu_info()

## 1. Datos limpios (los deja el notebook 01)

In [ ]:
X, y, classes, tr_idx, va_idx = fg.load_clean_data()
print(X.shape, classes)
print("train:", np.bincount(y[tr_idx]), "| val:", np.bincount(y[va_idx]))
sampler = fg.BatchSampler(X, y, tr_idx)

## 2. Modelo + pesos preentrenados

In [ ]:
resume = fg.has_checkpoint()
model = fg.build_model(len(classes)).to("cuda")
print(f"Parámetros: {fg.n_params(model):.0f} M | tamaño: {cfg.MODEL_SIZE} | reanudando: {resume}")
if not resume and cfg.MODEL_SIZE == "base":
    fg.load_pretrained(model, fg.get_pretrained_path())     # 1.ª vez: descarga ~712 MB

## 3. Benchmark: ¿qué micro-batch cabe y cuánto tardará?
Prueba micro-batches de mayor a menor y mide segundos por paso. **Mira el ETA** y ajusta `cfg.TOTAL_STEPS` si hace falta (p. ej. `cfg.TOTAL_STEPS = 1500`).

In [ ]:
micro, sec, peak = fg.find_micro_batch(model, sampler, total_gb)
accum = cfg.EFFECTIVE_BATCH // micro
sec_step = sec * accum
print(f"\nmicro-batch {micro} x acumulación {accum} = batch {micro * accum}")
print(f"~{sec_step:.1f} s por paso | {cfg.TOTAL_STEPS} pasos ≈ {sec_step * cfg.TOTAL_STEPS / 3600:.1f} h "
      f"≈ {cfg.TOTAL_STEPS * cfg.EFFECTIVE_BATCH / len(tr_idx):.0f} épocas")

## 4. Entrenar (reanudable)
Se detiene solo al llegar a `TOTAL_STEPS` o a `SESSION_MINUTES`. Guarda en Drive cada `SAVE_EVERY_MIN` minutos y al terminar. **Si Colab se desconecta: reconecta, ejecuta las celdas 1 a 4 de nuevo y esta celda continúa desde el último checkpoint.**

In [ ]:
opt = torch.optim.AdamW(fg.param_groups(model, cfg.WEIGHT_DECAY), lr=cfg.LR, betas=cfg.BETAS)
scaler = torch.amp.GradScaler("cuda")
state = fg.try_resume(model, opt, scaler)

In [ ]:
fg.train_loop(model, opt, scaler, sampler, (X, y, classes, va_idx), micro, accum, state)
fg.plot_history(state["history"])

## 5. Vista rápida de calidad
Una imagen por órgano con pocas iteraciones (rápido). La evaluación seria está en el notebook 03.

In [ ]:
from fg_eval import show_grid
imgs = fg.generate_many(model, list(range(len(classes))), bs=len(classes), seed=0,
                        cfg_scale=cfg.CFG, num_iter_list=(32, 8, 8))
show_grid(imgs, classes, ncols=len(classes), size=3.6)

### Si algo sale mal
| Síntoma | Qué hacer |
|---|---|
| `CUDA out of memory` en el benchmark | `cfg.MICRO_BATCH = 1` o `cfg.EFFECTIVE_BATCH = 16`; reinicia el entorno y repite |
| Descarga del preentrenado falla | descárgalo a mano (enlace en el README del repo) a `.../pretrained/fractalmar_base_in256.pth`, o `cfg.MODEL_SIZE = "small"` |
| `loss` = NaN / muchos pasos omitidos | baja `cfg.LR` a 1e-5 |
| val loss sube mientras train baja | sobreajuste: usa `best_fp16.pth` (ya se guarda) y/o menos pasos |
| val loss casi no baja | sube `cfg.LR` a 4e-5 o más pasos |
| Sesión se acaba | `last.pth` está en Drive; reconecta y reejecuta |